# 🛍️ Customer Segmentation using K-Means Clustering

Complete Week 4 major-project workflow using the Mall Customers dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

df = pd.read_csv('Mall_Customers.csv')
df.head()

## 1. Data Inspection and Cleaning

In [ ]:
print(df.info())
print(df.isnull().sum())

features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
data = df[features].copy()
for col in features:
    data[col] = pd.to_numeric(data[col], errors='coerce')
data = data.dropna().drop_duplicates()
data.describe()

## 2. Feature Scaling

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(data)
X[:5]

## 3. Elbow Method and Silhouette Score

In [ ]:
inertia = []
silhouette = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X, labels))

plt.figure(figsize=(8,5))
plt.plot(K_range, inertia, marker='o')
plt.title('Elbow Method')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.show()

plt.figure(figsize=(8,5))
plt.plot(K_range, silhouette, marker='o')
plt.title('Silhouette Score')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.show()

## 4. Final K-Means Model (K = 3)

In [ ]:
K = 3
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
data['Cluster'] = kmeans.fit_predict(X)

avg_spending = data.groupby('Cluster')['Spending Score (1-100)'].mean().sort_values()
ordered_clusters = list(avg_spending.index)
            
label_names = ['Low Spenders', 'Medium Spenders', 'High Spenders']
label_map = {cluster: label_names[i] for i, cluster in enumerate(ordered_clusters)}
data['Segment'] = data['Cluster'].map(label_map)
data.head()

## 5. Cluster Profiles

In [ ]:
profile = data.groupby(['Cluster', 'Segment'])[features].mean().round(2)
profile

## 6. Customer Groups by Income and Spending

In [ ]:
plt.figure(figsize=(8,5))
for segment in data['Segment'].unique():
    part = data[data['Segment'] == segment]
    plt.scatter(part['Annual Income (k$)'], part['Spending Score (1-100)'], label=segment, alpha=0.75)
            
plt.title('Customer Groups: Income vs Spending Score')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.show()

## 7. Customer Groups by Age and Spending

In [ ]:
plt.figure(figsize=(8,5))
for segment in data['Segment'].unique():
    part = data[data['Segment'] == segment]
    plt.scatter(part['Age'], part['Spending Score (1-100)'], label=segment, alpha=0.75)
            
plt.title('Customer Groups: Age vs Spending Score')
plt.xlabel('Age')
plt.ylabel('Spending Score (1-100)')
plt.legend()
plt.show()

## 8. PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8,5))
for segment in data['Segment'].unique():
    mask = data['Segment'] == segment
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=segment, alpha=0.75)

plt.title('PCA Visualization of Customer Segments')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.show()

In [ ]:
result = df.loc[data.index].copy()
result['Cluster'] = data['Cluster'].values
result['Segment'] = data['Segment'].values
result.to_csv('customer_segments.csv', index=False)
print('Saved customer_segments.csv')